# **Return of the Schema** for *Apulia Travel KG* 

## Path Definition Basic Elements

In [1]:
from rdflib import Graph, RDF, RDFS, OWL, Namespace
from urllib.parse import quote
from rdflib.namespace import split_uri
from rdflib.term import URIRef
from pathlib import Path
import pickle
import csv
import ast
import json

def serialize(graph, path):
    graph.serialize(path.with_suffix(".xml"), format="xml")
    !/home/anisa_bakiu/robot/robot.jar merge --input {path.with_suffix(".xml")} --output {path.with_suffix(".owl")}
    path.with_suffix(".xml").unlink()

In [5]:
MATERIALIZE = True
DATASET_NAME = "APULIATRAVEL"
DATASET_NAME += f"-{'MATERIALIZE' if MATERIALIZE else 'BASE'}"

home_path = Path().cwd().absolute().parent.parent 
dataset_path = home_path / "kgsaf_data" / f"{'materialize' if MATERIALIZE else 'base'}" / "unpack" / DATASET_NAME
onto_path = home_path / "kgsaf_data" / "ontologies"/ "unpack" / "APULIATRAVEL"

print("Base Path", home_path)
print("Ontology", onto_path)
print("Dataset", dataset_path)

print("")

if MATERIALIZE:
    print("Loading MATERIALIZED Ontology")
    onto_file = onto_path / "apulia_travel_merged_materialized.owl"
else:
    print("Loading BASE Ontology")
    onto_file = onto_path / "apulia_travel_merged.owl"

print("\tLoading Ontology")

apulia_onto = Graph()
apulia_onto.parse(onto_file)

print("\tOntology Loaded")

Base Path /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder
Ontology /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/ontologies/unpack/APULIATRAVEL
Dataset /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE

Loading MATERIALIZED Ontology
	Loading Ontology
	Ontology Loaded


# [O] ABOX Triple Cleaning

Removal of triples with individuals that are also classes

In [6]:
data_triples = Graph()
data_triples.parse(onto_path / "ApuliaTravelABox.ttl")

<Graph identifier=Nac16385c0cf8406d8f155a87b8ca7c6e (<class 'rdflib.graph.Graph'>)>

In [7]:
predicates = set(data_triples.predicates())
len(predicates)

40

In [ ]:
final_pred = set()

for pred in predicates:
    if (pred, RDF.type, OWL.ObjectProperty) in apulia_onto:
        final_pred.add(pred)
    else:
        print(f"Removing {pred}")

Removing https://w3id.org/italia/onto/SM/reviewAspect
Removing http://www.w3.org/1999/02/22-rdf-syntax-ns#type
Removing https://w3id.org/italia/onto/CLV/coordinate
Removing https://w3id.org/italia/onto/CLV/closes
Removing https://w3id.org/italia/onto/SM/emailAddress
Removing https://w3id.org/italia/onto/SM/userRating
Removing https://w3id.org/italia/onto/SM/isGeometryFor
Removing https://w3id.org/italia/onto/CLV/lat
Removing https://w3id.org/italia/onto/CLV/streetNumber
Removing http://www.w3.org/2000/01/rdf-schema#label
Removing https://w3id.org/italia/onto/CLV/opens
Removing https://w3id.org/italia/onto/CLV/long
Removing https://w3id.org/italia/onto/CLV/acronym
Removing https://w3id.org/italia/onto/l0/name
Removing https://w3id.org/italia/onto/CLV/postCode


In [10]:
print(len(final_pred))

25


In [12]:
out_graph = Graph()

with open(onto_path / "tsv_triples.tsv", "w") as f:
    for s,p,o in data_triples:
        if (p in final_pred):
            if (not ((s, RDF.type, OWL.Class) in apulia_onto)) and (not ((o, RDF.type, OWL.Class) in apulia_onto)):
                out_graph.add((s,p,o))
                f.write(f"{str(s)}\t{str(p)}\t{str(o)}\n")

out_graph.serialize(onto_path / "apulia_clean_abox.nt", format="nt")

/home/anisa_bakiu/anaconda3/envs/dlenv/lib/python3.11/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


<Graph identifier=N57418aff44f64e49b2a3e50d4dd693ba (<class 'rdflib.graph.Graph'>)>

In [13]:
import pykeen
from pykeen.triples import TriplesFactory
from pykeen.triples.splitting import CoverageSplitter
import numpy as np


triples = TriplesFactory.from_path(onto_path / "tsv_triples.tsv")

triples

/home/anisa_bakiu/anaconda3/envs/dlenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=76943, path="/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/ontologies/unpack/APULIATRAVEL/tsv_triples.tsv")

In [14]:
entity_mappings = {v:k for k,v in triples.entity_id_to_label.items()}
relation_mappings = {v:k for k,v in triples.relation_id_to_label.items()}

In [15]:
train, valid, test = triples.split(
    ratios=[0.85, 0.05, 0.1],
    random_state=42,
    method=CoverageSplitter(),      
)

In [16]:
train_clean = TriplesFactory.from_labeled_triples(
    triples=train.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

valid_clean = TriplesFactory.from_labeled_triples(
    triples=valid.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

test_clean = TriplesFactory.from_labeled_triples(
    triples=test.triples,
    entity_to_id=entity_mappings,
    relation_to_id=relation_mappings
)

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


In [17]:
print(train_clean)
print(test_clean)
print(valid_clean)

TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=65401)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=7695)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=3847)


In [18]:
from pykeen.triples.leakage import unleak

train_unleak, valid_unleak, test_unleak = unleak(
    train_clean,
    *[valid_clean, test_clean],
    n=None,
    minimum_frequency=0.97
    )

In [19]:
print(train_unleak)
print(test_unleak)
print(valid_unleak)

TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=65401)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=7695)
TriplesFactory(num_entities=29767, num_relations=25, create_inverse_triples=False, num_triples=3847)


In [23]:
from rdflib import Graph, URIRef

# Assicurati che la directory dei file esista
(dataset_path / "abox/splits").mkdir(parents=True, exist_ok=True)

targets = [
    ("train", train_unleak.triples),
    ("valid", valid_unleak.triples),
    ("test", test_unleak.triples),
]

for name, split in targets:
    out_graph = Graph()
    for s, p, o in split:
        out_graph.add((URIRef(s), URIRef(p), URIRef(o)))

    # Path finale per lo split
    out_path = dataset_path / "abox/splits" / f"{name}.nt"
    print("Saving:", out_path)
    out_graph.serialize(out_path, format="nt")

# Ora concateno tutti in triples.nt
triples_path = dataset_path / "abox" / "triples.nt"
(dataset_path / "abox").mkdir(exist_ok=True)
!cat {dataset_path}/abox/splits/*.nt > {triples_path}

print("Created triples.nt at:", triples_path)

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


Saving: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/splits/train.nt


/home/anisa_bakiu/anaconda3/envs/dlenv/lib/python3.11/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


Saving: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/splits/valid.nt
Saving: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/splits/test.nt
Created triples.nt at: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/triples.nt


In [24]:

del out_graph
del data_triples

# ABOX Individuals and Class Assertions

In [25]:
data_triples = Graph()
data_triples.parse(dataset_path / "abox" / "triples.nt")

individuals = set(data_triples.subjects()) | set(data_triples.objects())

print("Len Individuals", len(individuals))
del data_triples

Len Individuals 29767


In [27]:
from rdflib import Graph, RDF, OWL, URIRef
from pathlib import Path
import subprocess

def serialize_with_robot(graph: Graph, path: Path, robot_jar_path: Path):
    """
    Salva il grafo RDF in XML e poi usa Robot per creare un file OWL.
    
    Parameters:
    - graph: rdflib.Graph da salvare
    - path: Path del file senza estensione
    - robot_jar_path: Path al robot.jar
    """
    # Salva come XML temporaneo
    xml_path = path.with_suffix(".xml")
    graph.serialize(xml_path, format="xml")
    print(f"Saved temporary XML: {xml_path}")

    # Esegui Robot merge
    owl_path = path.with_suffix(".owl")
    subprocess.run([
        "java", "-jar", str(robot_jar_path),
        "merge",
        "--input", str(xml_path),
        "--output", str(owl_path)
    ], check=True)
    print(f"Created OWL file: {owl_path}")

    # Rimuovi XML temporaneo
    xml_path.unlink()
    print(f"Removed temporary XML: {xml_path}")

# -----------------------------
# Uso concreto
# -----------------------------

out_graph = Graph()
for ind in individuals:
    out_graph.add((ind, RDF.type, OWL.NamedIndividual))

# Path per il file OWL
individuals_path = dataset_path / "abox" / "individuals"

# Path al tuo robot.jar
robot_jar_path = Path("/home/anisa_bakiu/robot/robot.jar")

serialize_with_robot(out_graph, individuals_path, robot_jar_path)
del out_graph

Saved temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/individuals.xml
Created OWL file: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/individuals.owl
Removed temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/individuals.xml


In [28]:
ca_graph = Graph()
ca_graph.parse(onto_path / "ApuliaTravelABox.ttl")

<Graph identifier=N9e09dd3a04484dcda89b1b3037a7aac5 (<class 'rdflib.graph.Graph'>)>

### [BASE] RDF Lib Class Assertions

In [31]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef
from pathlib import Path
import subprocess

# --- Definizione URI di sistema da escludere ---
BUILTIN_URI = {
    RDF.type,        # http://www.w3.org/1999/02/22-rdf-syntax-ns#type
    RDFS.Class,      # http://www.w3.org/2000/01/rdf-schema#Class
    OWL.Class,       # http://www.w3.org/2002/07/owl#Class
    OWL.Thing,       # http://www.w3.org/2002/07/owl#Thing
    RDFS.Resource    # http://www.w3.org/2000/01/rdf-schema#Resource
}

# --- Funzione serialize usando ROBOT jar ---
def serialize(graph, path):
    xml_file = path.with_suffix(".xml")
    owl_file = path.with_suffix(".owl")
    
    # Salva XML temporaneo
    graph.serialize(xml_file, format="xml")
    print(f"Saved temporary XML: {xml_file}")
    
    # Chiama ROBOT jar per creare OWL
    jar_path = "/home/anisa_bakiu/robot/robot.jar"
    subprocess.run([
        "java", "-jar", jar_path,
        "merge",
        "--input", str(xml_file),
        "--output", str(owl_file)
    ], check=True)
    print(f"Created OWL file: {owl_file}")
    
    # Rimuovi XML temporaneo
    xml_file.unlink()
    print(f"Removed temporary XML: {xml_file}")

# --- Creazione del grafo delle class assertions ---
out_graph = Graph()

for ind in individuals:
    for ca in set(ca_graph.objects(ind, RDF.type)) - BUILTIN_URI:
        if (ca, RDF.type, OWL.Class) in apulia_onto:
            out_graph.add((ind, RDF.type, ca))
        else:
            print(f"Not a Class {ca}")

# --- Serializzazione sicura nella cartella desiderata ---
out_path = dataset_path / "abox" / "class_assertions"
out_path.parent.mkdir(parents=True, exist_ok=True)  # assicura che la cartella esista
serialize(out_graph, out_path)

# --- Pulizia ---
del out_graph

Saved temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.xml
Created OWL file: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.owl
Removed temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.xml


### [REASONED] Reasoner Class Assertions

In [32]:
out_graph = Graph()


for ind in individuals:
    for ca in  set(ca_graph.objects(ind, RDF.type)) - BUILTIN_URI:
        if (ca, RDF.type, OWL.Class) in apulia_onto:
            out_graph.add((ind, RDF.type, ca))
        else:
            print(f"Not a Class {ca}")

serialize(out_graph, dataset_path / "abox" / "unreasoned_class_assertions")
del out_graph
     

Saved temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/unreasoned_class_assertions.xml
Created OWL file: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/unreasoned_class_assertions.owl
Removed temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/unreasoned_class_assertions.xml


In [37]:
import subprocess
from pathlib import Path

# Paths
dataset_path = Path("/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE")
apulia_path = Path("/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/ontologies/unpack/APULIATRAVEL")

robot_jar = "/home/anisa_bakiu/robot/robot.jar"

# --- MERGE ---
merge_cmd = [
    "java", "-Xmx16G", "-jar", robot_jar, "merge", "-vvv",
    "--input", str(dataset_path / "abox" / "unreasoned_class_assertions.owl"),
    "--input", str(dataset_path / "abox" / "individuals.owl"),
    "--input", str(dataset_path / "abox" / "triples.nt"),
    "--input", str(apulia_path / "apulia_travel_merged_materialized.owl"),
    "--output", str(dataset_path / "abox" / "intermediate_abox_tbox.owl")
]

subprocess.run(merge_cmd, check=True)

# --- REASON ---
reason_cmd = [
    "java", "-Xmx16G", "-jar", robot_jar, "reason", "-vvv",
    "--reasoner", "HermiT",
    "--create-new-ontology", "true",
    "--input", str(dataset_path / "abox" / "intermediate_abox_tbox.owl"),
    "--output", str(dataset_path / "abox" / "inferred_class_assertions.owl"),
    "--axiom-generators", "ClassAssertion",
    "--remove-redundant-subclass-axioms", "false",
    "--exclude-tautologies", "structural",
    "--include-indirect", "true",
    "-D", str(dataset_path / "class_assertions_debug.owl")
]

subprocess.run(reason_cmd, check=True)

2025-12-26 21:04:05,878 DEBUG org.obolibrary.robot.IOHelper - Loading ontology /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/unreasoned_class_assertions.owl with catalog file null
2025-12-26 21:04:05,883 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading file META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2025-12-26 21:04:05,883 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/anisa_bakiu/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2025-12-26 21:04:05,884 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/anisa_bakiu/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.model.OWLOntologyManager
2025-12-26 21:04:05,884 DEBUG org.semanticweb.owlapi.utilities.Injector - Loading URL for service jar:file:/home/anisa_bakiu/robot/robot.jar!/META-INF/services/org.semanticweb.owlapi.

CompletedProcess(args=['java', '-Xmx16G', '-jar', '/home/anisa_bakiu/robot/robot.jar', 'reason', '-vvv', '--reasoner', 'HermiT', '--create-new-ontology', 'true', '--input', '/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/intermediate_abox_tbox.owl', '--output', '/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/inferred_class_assertions.owl', '--axiom-generators', 'ClassAssertion', '--remove-redundant-subclass-axioms', 'false', '--exclude-tautologies', 'structural', '--include-indirect', 'true', '-D', '/home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/class_assertions_debug.owl'], returncode=0)

In [38]:
ca = Graph()
ca.parse(dataset_path / "abox" / "unreasoned_class_assertions.owl")
ca.parse(dataset_path / "abox" / "inferred_class_assertions.owl")

<Graph identifier=N28e2caa7dfd04549b79cd72b811b60e9 (<class 'rdflib.graph.Graph'>)>

In [39]:
out_graph = Graph()

for ind in individuals:
    for o in set(ca.objects(ind, RDF.type)) - BUILTIN_URI:
        out_graph.add((ind,RDF.type, o))

serialize(out_graph, dataset_path / "abox" / "class_assertions")

Saved temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.xml
Created OWL file: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.owl
Removed temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/abox/class_assertions.xml


In [40]:
(dataset_path / "abox" / "inferred_class_assertions.owl").unlink()
(dataset_path / "abox" / "unreasoned_class_assertions.owl").unlink()
(dataset_path / "abox" / "intermediate_abox_tbox.owl").unlink()

In [41]:
del(out_graph)
del(ca)

# TBOX and RBOX Extraction

In [42]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from rdflib import Graph, URIRef, BNode, RDF, RDFS, OWL
import  utils.conventions.paths as pc
from utils.conventions.builtins import BUILTIN_URIS

class SignatureModularizer:
    def __init__(self, schema, seed):
        self.schema = schema
        self.seed = seed

    def modularize(self):
        return self._extract_recursive_description()

    def _extract_recursive_description(self) -> Graph:

        extracted_graph = Graph()
        elem_to_process = set(self.seed)
        processed = set()

        while elem_to_process:

            e = elem_to_process.pop()
            processed.add(e)

            print(f"Processing {e}")

            for s,p,o in self.schema.triples((e, None, None)):
                extracted_graph.add((s,p,o))

                if (o not in BUILTIN_URIS) and (o not in processed):

                    if isinstance(o, BNode):
                        elem_to_process.add(o)

                    if (o, RDF.type, OWL.Class) in self.schema:
                        elem_to_process.add(o)

                    if (o, RDF.type, OWL.ObjectProperty) in self.schema:
                        elem_to_process.add(o)

                    if (o, RDF.type, OWL.DatatypeProperty) in self.schema:
                        elem_to_process.add(o)

        return extracted_graph
    

class SchemaDecomposition:
    def __init__(self, input_graph):
        self.onto_graph = input_graph
    
    def decompose(self):
        return self._rbox_decompose(), self._taxonomy_decompose(), self._schema_decompose()


    def _rbox_decompose(self):
        rbox_graph = Graph()
        for prop in set(self.onto_graph.subjects(RDF.type, OWL.ObjectProperty)) - BUILTIN_URI:
            rbox_graph += self._extract_description(prop)

        for prop in set(self.onto_graph.subjects(RDF.type, OWL.DatatypeProperty)) - BUILTIN_URI:
            rbox_graph += self._extract_description(prop)
        return rbox_graph


    def _taxonomy_decompose(self):
        taxonomy_graph = Graph()

        for c in set(self.onto_graph.subjects(RDF.type, OWL.Class)) - BUILTIN_URI:
            for s,p,o in self.onto_graph.triples((c, None, None)):
                if p == RDFS.subClassOf:
                    taxonomy_graph.add((s,p,o))
                    if isinstance(o, BNode):
                        taxonomy_graph += self._extract_description(o)

        return taxonomy_graph

    def _schema_decompose(self):
        schema_graph = Graph()

        for c in set(self.onto_graph.subjects(RDF.type, OWL.Class)) - BUILTIN_URI:
            if not isinstance(c, BNode):
                for s,p,o in self.onto_graph.triples((c, None, None)):
                    if p != RDFS.subClassOf:
                        
                        schema_graph.add((s,p,o))

                        for elem in self.onto_graph.objects(o, RDF.type):
                            schema_graph.add((o, RDF.type, elem))

                        if isinstance(o, BNode):
                            print(f"Found BNODE in Triple {s, p, o}")
                            schema_graph += self._extract_description(o)

        return schema_graph

    def _extract_description(self, elem: URIRef) -> Graph:

        extracted_graph = Graph()
        elem_to_process = {elem}
        processed = set()

        while elem_to_process:

            e = elem_to_process.pop()
            processed.add(e)

            print(f"Processing {e}")

            for s,p,o in self.onto_graph.triples((e, None, None)):
                extracted_graph.add((s,p,o))

                if (o not in BUILTIN_URI) and (o not in processed):
                    if isinstance(o, BNode):
                        elem_to_process.add(o)

                    if (o, RDF.type, OWL.Class) in self.onto_graph:
                        extracted_graph.add((o, RDF.type, OWL.Class))

                    if (o, RDF.type, OWL.ObjectProperty) in self.onto_graph:
                        extracted_graph.add((o, RDF.type, OWL.ObjectProperty))

                    if (o, RDF.type, OWL.DatatypeProperty) in self.onto_graph:
                        extracted_graph.add((o, RDF.type, OWL.DatatypeProperty))

        return extracted_graph

In [43]:
data_triples = Graph()
data_triples.parse(dataset_path / "abox" / "triples.nt")

class_assertions = Graph()
class_assertions.parse(dataset_path / "abox" / "class_assertions.owl")

<Graph identifier=Nfed3dd2f081e4ca68a8a171c2dfb37bd (<class 'rdflib.graph.Graph'>)>

In [44]:
seed_obj_props = set(data_triples.predicates())
print("Seed Object Properties", len(seed_obj_props))

seed_classes =  set(class_assertions.subjects(RDF.type, OWL.Class))
print("Seed Classes", len(seed_classes))

Seed Object Properties 25
Seed Classes 65


In [45]:

modularizer = SignatureModularizer(apulia_onto, seed_classes | seed_obj_props)
out_graph = modularizer.modularize()

serialize(out_graph, dataset_path / "ontology")

Processing https://w3id.org/italia/onto/TI/TemporalEntity
Processing https://w3id.org/italia/onto/SM/isWebSiteOf
Processing https://w3id.org/italia/onto/SM/hasReview
Processing https://apuliatravel.org/td/Mill
Processing https://apuliatravel.org/td/NationalPark
Processing https://w3id.org/italia/onto/l0/Entity
Processing https://w3id.org/italia/onto/SM/OnlineContactPoint
Processing https://apuliatravel.org/td/Museum
Processing https://w3id.org/italia/onto/CLV/hasProvince
Processing https://w3id.org/italia/onto/l0/Object
Processing https://apuliatravel.org/td/Convent
Processing https://w3id.org/italia/onto/POI/PointOfInterest
Processing https://w3id.org/italia/onto/SM/hasContactPoint
Processing https://w3id.org/italia/onto/TI/hasDayOfWeek
Processing https://w3id.org/italia/onto/SM/hasEmail
Processing https://w3id.org/italia/onto/CLV/hasAddressComponent
Processing https://apuliatravel.org/td/Church
Processing https://apuliatravel.org/td/WWFOasis
Processing Ncf95429f447a41319f4ebb088963aa

In [47]:
from rdflib import Graph
from pathlib import Path

# Funzione robusta di serializzazione
def serialize(graph, path: Path):
    # Assicuriamoci che la cartella esista
    path.parent.mkdir(parents=True, exist_ok=True)

    xml_file = path.with_suffix(".xml")
    owl_file = path.with_suffix(".owl")

    # Salva XML temporaneo
    graph.serialize(xml_file, format="xml")
    print(f"Saved temporary XML: {xml_file}")

    # Chiama ROBOT jar per creare OWL
    !java -Xmx16G -jar /home/anisa_bakiu/robot/robot.jar merge --input {xml_file} --output {owl_file}

    # Rimuovi XML temporaneo
    xml_file.unlink()
    print(f"Created OWL file: {owl_file}")

# Carica l'ontologia principale
onto_graph = Graph()
onto_graph.parse(dataset_path / "ontology.owl")

# Decomposizione dello schema
decomposer = SchemaDecomposition(onto_graph)
rbox_graph, taxonomy_graph, schema_graph = decomposer.decompose()

# Salvataggio dei grafi decompositi
serialize(rbox_graph, dataset_path / "rbox" / "roles")
serialize(taxonomy_graph, dataset_path / "tbox" / "taxonomy")
serialize(schema_graph, dataset_path / "tbox" / "schema")

Processing https://w3id.org/italia/onto/CLV/isAddressInTimeFor
Processing https://w3id.org/italia/onto/SM/isWebSiteOf
Processing https://w3id.org/italia/onto/SM/hasReview
Processing https://w3id.org/italia/onto/TI/hasTimeInstantInside
Processing https://w3id.org/italia/onto/SM/hasCreator
Processing Nca850aa616f5460b888cde4ec67f9289
Processing https://w3id.org/italia/onto/CLV/isGeometryFor
Processing https://w3id.org/italia/onto/POI/isPOICategoryFor
Processing https://w3id.org/italia/onto/SM/isUserAccountOf
Processing https://w3id.org/italia/onto/SM/isTelephoneOf
Processing https://w3id.org/italia/onto/CLV/hasProvince
Processing https://w3id.org/italia/onto/TI/isDayOfWeekOf
Processing N2cb3152915e94a8f9d10b8a3c60aff4e
Processing https://w3id.org/italia/onto/SM/hasContactPoint
Processing https://w3id.org/italia/onto/TI/hasDayOfWeek
Processing Nd3e884dda3784389ad2feea817972f01
Processing Nee3ea5c4c254424d997c077c78c4c929
Processing https://w3id.org/italia/onto/POI/isPOINameInTimeFor
Proce

In [48]:
from rdflib import BNode





def extract_description(graph: Graph, elem: URIRef) -> Graph:

    extracted_graph = Graph()
    elem_to_process = {elem}
    processed = set()


    while elem_to_process:

        e = elem_to_process.pop()
        processed.add(e)

        print(f"Processing {e}")

        for s,p,o in graph.triples((e, None, None)):
            extracted_graph.add((s,p,o))

            if (o not in BUILTIN_URI) and (o not in processed):
                if isinstance(o, BNode):
                    elem_to_process.add(o)

                if (o, RDF.type, OWL.Class) in graph:
                    extracted_graph.add((o, RDF.type, OWL.Class))

                if (o, RDF.type, OWL.ObjectProperty) in graph:
                    extracted_graph.add((o, RDF.type, OWL.ObjectProperty))

                if (o, RDF.type, OWL.DatatypeProperty) in graph:
                    extracted_graph.add((o, RDF.type, OWL.DatatypeProperty))

    return extracted_graph


rbox_graph = Graph()
for prop in set(onto_graph.subjects(RDF.type, OWL.ObjectProperty)) - BUILTIN_URI:
    rbox_graph += extract_description(onto_graph, prop)

for prop in set(onto_graph.subjects(RDF.type, OWL.DatatypeProperty)) - BUILTIN_URI:
    rbox_graph += extract_description(onto_graph, prop)


serialize(rbox_graph, dataset_path / "rbox" / "roles")

Processing https://w3id.org/italia/onto/CLV/isAddressInTimeFor
Processing https://w3id.org/italia/onto/SM/isWebSiteOf
Processing https://w3id.org/italia/onto/SM/hasReview
Processing https://w3id.org/italia/onto/TI/hasTimeInstantInside
Processing https://w3id.org/italia/onto/SM/hasCreator
Processing Nca850aa616f5460b888cde4ec67f9289
Processing https://w3id.org/italia/onto/CLV/isGeometryFor
Processing https://w3id.org/italia/onto/POI/isPOICategoryFor
Processing https://w3id.org/italia/onto/SM/isUserAccountOf
Processing https://w3id.org/italia/onto/SM/isTelephoneOf
Processing https://w3id.org/italia/onto/CLV/hasProvince
Processing https://w3id.org/italia/onto/TI/isDayOfWeekOf
Processing N2cb3152915e94a8f9d10b8a3c60aff4e
Processing https://w3id.org/italia/onto/SM/hasContactPoint
Processing https://w3id.org/italia/onto/TI/hasDayOfWeek
Processing Nd3e884dda3784389ad2feea817972f01
Processing Nee3ea5c4c254424d997c077c78c4c929
Processing https://w3id.org/italia/onto/POI/isPOINameInTimeFor
Proce

In [49]:
taxonomy_graph = Graph()

for c in set(onto_graph.subjects(RDF.type, OWL.Class)) - BUILTIN_URI:
    for s,p,o in onto_graph.triples((c, None, None)):
        if p == RDFS.subClassOf:
            taxonomy_graph.add((s,p,o))
            if isinstance(o, BNode):
                taxonomy_graph += extract_description(onto_graph, o)

serialize(taxonomy_graph, dataset_path / "tbox" / "taxonomy")

Processing Ne225010850264f6cbecfe963b995c1f7
Processing N16d998b0d893432aa75b2680d900b9e9
Processing Ne22368ec807f4fbd9043e0ebe28e3e1d
Processing N8a328dded97c4271bf569f87045b663e
Processing N72ecfe2da9d14b68aa2c538ef7ec4cb8
Processing N9bfba6a3a1844b8aa052de47defbb25e
Processing N886728e9fc524b279e403edb4a22e8e6
Processing Nf5a75001042947258ef9085164f2e89b
Processing Nbae6fbc6be87404ab1404a291119c235
Processing N675189ff0a674cae8ba861e4d5f2652c
Processing Na62e2de597e9413292096387ddda9140
Processing Nae8639dbce944bbeac484c091c4d505d
Processing Ne6e28a3bb29542ee943ec33d1f676014
Processing N69255bf3887e4953a4bd51c960ec01e4
Processing Nf46d196c50b64466b14f930349343d61
Processing N7326abf4f9b04adea63b944ad148f29e
Processing N10bdc831afce4b329ad4f2a877a231a2
Processing N03c3951fe8ef42b78226f89205e81b91
Processing N237490ef1914434698aaedddad073cc9
Processing N11076459054245959be99712c2ccb6db
Processing N3bad60a1023444f78b46851dd0d55982
Processing Neda3bab888884c328da2bf9414199dcb
Processing

In [50]:
schema_graph = Graph()


for c in set(onto_graph.subjects(RDF.type, OWL.Class)) - BUILTIN_URI:
    if not isinstance(c, BNode):
        for s,p,o in onto_graph.triples((c, None, None)):
            if p != RDFS.subClassOf:
                
                schema_graph.add((s,p,o))

                for elem in onto_graph.objects(o, RDF.type):
                    schema_graph.add((o, RDF.type, elem))

                if isinstance(o, BNode):
                    print(f"Found BNODE in Triple {s, p, o}")
                    schema_graph += extract_description(onto_graph, o)
            

serialize(schema_graph, dataset_path / "tbox" / "schema")

Saved temporary XML: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/tbox/schema.xml
Created OWL file: /home/anisa_bakiu/Downloads/Progetto_DL/kg-saf-folder/kgsaf_data/materialize/unpack/APULIATRAVEL-MATERIALIZE/tbox/schema.owl


# Final Ontology and Knowledge Graph

In [51]:
from pathlib import Path

# Assicuriamoci che la cartella di output esista
(dataset_path / "abox").mkdir(parents=True, exist_ok=True)

# Merge ontologie con ROBOT
!java -Xmx16G -jar /home/anisa_bakiu/robot/robot.jar merge \
    --input {dataset_path / "ontology.owl"} \
    --input {dataset_path / "abox" / "individuals.owl"} \
    --input {dataset_path / "abox" / "triples.nt"} \
    --input {dataset_path / "abox" / "class_assertions.owl"} \
    --output {dataset_path / "knowledge_graph.owl"}